# OASIS-1 3D Benchmark Visualization

Notebook tự chứa toàn bộ code. Cell setup chỉ load dữ liệu và helper; mỗi code cell sau đó tạo đúng một hình và lưu vào `visualize/oasis1_3d_functional_20260527_025659/`.

In [ ]:
# Setup: load benchmark data and helper functions. Cell này không vẽ hình.
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates += [path / 'BioMedReg' for path in [cwd, *cwd.parents]]
    for path in candidates:
        if (path / 'src').is_dir() and (path / 'outputs').is_dir():
            return path.resolve()
    raise RuntimeError('Cannot find BioMedReg repo root. Run from BioMedReg or from its parent folder.')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
%cd $ROOT

from src.utils.io import read_image
from src.utils.metrics import normalize_image

benchmark_dir = Path('outputs/benchmark/oasis1_3d_functional_20260527_025659')
out_dir = Path('visualize') / benchmark_dir.name
out_dir.mkdir(parents=True, exist_ok=True)

method_order = ['classical', 'pso', 'voxelmorph', 'transmorph']
method_labels = {
    'classical': 'Classical',
    'pso': 'PSO',
    'voxelmorph': 'VoxelMorph',
    'transmorph': 'TransMorph',
}
method_colors = {
    'classical': '#4C78A8',
    'pso': '#F58518',
    'voxelmorph': '#54A24B',
    'transmorph': '#B279A2',
}
numeric_cols = [
    'run_seconds', 'before_mse', 'after_mse', 'delta_mse',
    'before_ncc', 'after_ncc', 'delta_ncc',
    'dice_before_mean', 'dice_after_mean', 'dice_delta_mean',
    'jacobian_folding_percent',
]

df = pd.read_csv(benchmark_dir / 'benchmark_results.csv')
df = df[df['success'].astype(bool)].copy()
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['pair_index'] = pd.to_numeric(df['pair_index']).astype(int)
methods = [m for m in method_order if m in set(df['method'])]

def save_current_figure(name, dpi=170):
    path = out_dir / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'Saved: {path}')

def mid_slice(volume, axis=0):
    idx = volume.shape[axis] // 2
    if axis == 0:
        return volume[idx, :, :]
    if axis == 1:
        return volume[:, idx, :]
    return volume[:, :, idx]

def as_display(image):
    return normalize_image(np.nan_to_num(image.astype(np.float32), nan=0.0))

def overlay_rgb(fixed, candidate):
    fixed_s = as_display(fixed)
    cand_s = as_display(candidate)
    rgb = np.zeros(fixed_s.shape + (3,), dtype=np.float32)
    rgb[..., 0] = fixed_s
    rgb[..., 1] = cand_s
    rgb[..., 2] = 0.25 * fixed_s
    return np.clip(rgb, 0.0, 1.0)

def dense_field_to_zyx(field, method):
    if method == 'classical':
        return field[..., [2, 1, 0]]
    return field

def jacobian_det_3d(field_zyx):
    gz0, gy0, gx0 = np.gradient(field_zyx[..., 0], edge_order=1)
    gz1, gy1, gx1 = np.gradient(field_zyx[..., 1], edge_order=1)
    gz2, gy2, gx2 = np.gradient(field_zyx[..., 2], edge_order=1)
    j00, j01, j02 = 1.0 + gz0, gy0, gx0
    j10, j11, j12 = gz1, 1.0 + gy1, gx1
    j20, j21, j22 = gz2, gy2, 1.0 + gx2
    return j00 * (j11 * j22 - j12 * j21) - j01 * (j10 * j22 - j12 * j20) + j02 * (j10 * j21 - j11 * j20)

print(f'Loaded {len(df)} successful test rows from {benchmark_dir}')
print(f'Writing figures to {out_dir}')

In [ ]:
# Figure 1: mean metric summary
grouped = df.groupby('method').mean(numeric_only=True).reindex(methods)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('OASIS-1 3D Functional Benchmark: Mean Test Metrics', fontsize=15)
plots = [
    ('after_mse', 'After MSE', 'lower is better'),
    ('after_ncc', 'After NCC', 'higher is better'),
    ('dice_after_mean', 'Dice after', 'higher is better'),
    ('run_seconds', 'Runtime / pair (s)', 'lower is better'),
]
for ax, (col, title, subtitle) in zip(axes.flat, plots):
    vals = grouped[col].to_numpy()
    bars = ax.bar([method_labels[m] for m in methods], vals, color=[method_colors[m] for m in methods])
    ax.set_title(f'{title}\n{subtitle}', fontsize=11)
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:.4f}', ha='center', va='bottom', fontsize=9)
save_current_figure('summary_metrics.png')
plt.show()

In [ ]:
# Figure 2: metric distributions across test pairs
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Metric Distributions Across 85 Test Pairs', fontsize=15)
plots = [
    ('after_mse', 'After MSE'),
    ('after_ncc', 'After NCC'),
    ('dice_after_mean', 'Dice after'),
    ('run_seconds', 'Runtime / pair (s)'),
    ('delta_ncc', 'NCC improvement'),
    ('jacobian_folding_percent', 'Jacobian folding %'),
]
for ax, (col, title) in zip(axes.flat, plots):
    data = [df.loc[df['method'] == m, col].dropna().to_numpy() for m in methods]
    box = ax.boxplot(data, patch_artist=True, widths=0.62)
    for patch, method in zip(box['boxes'], methods):
        patch.set_facecolor(method_colors[method])
        patch.set_alpha(0.72)
    ax.set_title(title)
    ax.set_xticklabels([method_labels[m] for m in methods], rotation=25)
    ax.grid(axis='y', alpha=0.25)
save_current_figure('metric_distributions.png')
plt.show()

In [ ]:
# Figure 3: runtime-quality tradeoff
grouped = df.groupby('method').mean(numeric_only=True).reindex(methods)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Runtime vs Registration Quality', fontsize=15)
for ax, y_col, y_label in [(axes[0], 'after_ncc', 'Mean after NCC'), (axes[1], 'dice_after_mean', 'Mean Dice after')]:
    for method in methods:
        x = grouped.loc[method, 'run_seconds']
        y = grouped.loc[method, y_col]
        ax.scatter(x, y, s=180, color=method_colors[method], edgecolor='black', linewidth=0.7)
        ax.annotate(method_labels[method], xy=(x, y), xytext=(7, 4), textcoords='offset points', fontsize=10)
    ax.set_xlabel('Mean runtime per pair (s)')
    ax.set_ylabel(y_label)
    ax.grid(alpha=0.25)
save_current_figure('runtime_quality_tradeoff.png')
plt.show()

In [ ]:
# Figure 4: Dice heatmap by anatomical label
records = []
for row in df.itertuples(index=False):
    path = benchmark_dir / 'test_runs' / row.method / f'pair_{int(row.pair_index):03d}' / 'benchmark_metrics.json'
    if not path.exists():
        continue
    payload = json.loads(path.read_text())
    dice_after = payload.get('label_metrics', {}).get('dice_after', {}) or {}
    for label, value in dice_after.items():
        records.append({'method': row.method, 'label': label, 'dice_after': value})
label_df = pd.DataFrame.from_records(records)
pivot = label_df.pivot_table(index='method', columns='label', values='dice_after', aggfunc='mean').reindex(methods)
pivot = pivot[pivot.mean(axis=0).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(16, 4.5))
im = ax.imshow(pivot.to_numpy(), aspect='auto', cmap='viridis', vmin=0.0, vmax=1.0)
ax.set_title('Mean Dice After Registration by Anatomical Label')
ax.set_yticks(np.arange(len(methods)))
ax.set_yticklabels([method_labels[m] for m in methods])
ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=65, ha='right', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label='Dice')
save_current_figure('label_dice_heatmap.png')
plt.show()

In [ ]:
# Figure 5: qualitative registration gallery for one selected test pair
pair_index = int(df.groupby('pair_index')['after_ncc'].mean().sort_values().index[len(df['pair_index'].unique()) // 2])
pair_df = df[df['pair_index'] == pair_index]
base = pair_df.iloc[0]
fixed = read_image(base['fixed'])
moving = read_image(base['moving'])
fixed_sl = mid_slice(fixed, axis=0)
moving_sl = mid_slice(moving, axis=0)

rows = [('Before', moving_sl)]
for method in methods:
    registered = read_image(pair_df[pair_df['method'] == method].iloc[0]['registered'])
    rows.append((method_labels[method], mid_slice(registered, axis=0)))

fig, axes = plt.subplots(len(rows), 4, figsize=(13, 3.0 * len(rows)))
fig.suptitle(f'Qualitative Registration Gallery: Test Pair {pair_index:03d}', fontsize=15)
for r, (row_label, candidate) in enumerate(rows):
    diff = np.abs(as_display(fixed_sl) - as_display(candidate))
    images = [
        (fixed_sl, 'Fixed', 'gray'),
        (candidate, 'Moving' if row_label == 'Before' else 'Registered', 'gray'),
        (overlay_rgb(fixed_sl, candidate), 'Overlay: fixed red, candidate green', None),
        (diff, '|fixed - candidate|', 'magma'),
    ]
    axes[r, 0].set_ylabel(row_label, fontsize=11)
    for c, (image, title, cmap) in enumerate(images):
        ax = axes[r, c]
        ax.imshow(image, cmap=cmap) if cmap else ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
save_current_figure(f'qualitative_pair_{pair_index:03d}.png', dpi=150)
plt.show()

In [ ]:
# Figure 6: dense deformation diagnostics for the same selected test pair
dense_methods = ['classical', 'voxelmorph', 'transmorph']
fig, axes = plt.subplots(len(dense_methods), 3, figsize=(12, 3.3 * len(dense_methods)))
fig.suptitle(f'Dense Deformation Diagnostics: Test Pair {pair_index:03d}', fontsize=15)
for r, method in enumerate(dense_methods):
    field_path = benchmark_dir / 'test_runs' / method / f'pair_{pair_index:03d}' / 'deformation_field.npy'
    field = dense_field_to_zyx(np.load(field_path), method)
    magnitude = np.linalg.norm(field, axis=-1)
    jac = jacobian_det_3d(field)
    fold = jac <= 0.0
    slices = [
        (mid_slice(magnitude, axis=0), 'Displacement magnitude', 'viridis'),
        (mid_slice(jac, axis=0), 'Jacobian determinant', 'coolwarm'),
        (mid_slice(fold.astype(np.float32), axis=0), 'Folding mask (J <= 0)', 'Reds'),
    ]
    axes[r, 0].set_ylabel(method_labels[method], fontsize=11)
    for c, (image, title, cmap) in enumerate(slices):
        im = axes[r, c].imshow(image, cmap=cmap)
        axes[r, c].set_title(title, fontsize=10)
        axes[r, c].set_xticks([])
        axes[r, c].set_yticks([])
        fig.colorbar(im, ax=axes[r, c], fraction=0.035, pad=0.02)
save_current_figure(f'deformation_pair_{pair_index:03d}.png', dpi=150)
plt.show()